# Etapa 4 — Llamada de prueba (DeepSeek V4.1 Flash, razonamiento desactivado)

Nota: el plan inicial (Llama 3.3 70B en un proveedor gratuito) no fue posible porque ese modelo fue retirado; se usa DeepSeek V4.1 Flash.

**UNA sola llamada**: el primer lote (los primeros 20 genes) de la red de hipertrofia, con temperatura 1.0, usando el prompt idéntico al del repo.
Comprobamos: (1) que la temperatura se envía y que el razonamiento está desactivado (si no, la temperatura no tendría efecto), (2) que el JSON sale válido,
(3) cuántas interacciones devuelve. La API key se lee desde `experimento/.env`; nunca se imprime.

In [1]:
import sys
sys.dont_write_bytecode = True
import json, pandas as pd
import llm_prompting_deepseek as gq

resumen = gq.llamada_de_prueba(red="hypertrophy", temperatura=1.0)
print(json.dumps({k: v for k, v in resumen.items() if k not in ("headers_limite",)}, indent=2, ensure_ascii=False))

{
  "proveedor": "deepseek",
  "modelo_pedido": "deepseek-flash",
  "modelo_devuelto": "deepseek-flash",
  "razonamiento_desactivado_en_peticion": {
    "thinking": {
      "type": "disabled"
    }
  },
  "hubo_razonamiento_en_respuesta": false,
  "temperatura_enviada": 1.0,
  "finish_reason": "stop",
  "json_valido": true,
  "error": null,
  "n_interacciones_validas": 37,
  "n_descartadas": 0,
  "forma_alternativa": false,
  "usage": {
    "completion_tokens": 944,
    "prompt_tokens": 1420,
    "total_tokens": 2364,
    "completion_tokens_details": null,
    "prompt_tokens_details": {
      "audio_tokens": null,
      "cached_tokens": 0
    },
    "prompt_cache_hit_tokens": 0,
    "prompt_cache_miss_tokens": 1420
  },
  "incidencias_api": [],
  "segundos": 4.24,
  "estado": "ok",
  "n_genes": 225,
  "max_conexiones": 14,
  "llamadas_por_replica": 12
}


In [2]:
print("Modelos que ve la cuenta:", gq.listar_modelos())
print("Cabeceras de límites devueltas por la API:", resumen["headers_limite"] or "(ninguna)")

Modelos que ve la cuenta: ['deepseek-flash', 'deepseek-v4-pro']
Cabeceras de límites devueltas por la API: (ninguna)


In [3]:
import json
from pathlib import Path
datos = json.loads((Path(gq.AQUI) / "resultados" / "prueba" / "prueba_respuesta.json").read_text(encoding="utf-8"))
df = pd.DataFrame(datos["interacciones"])
print("Interacciones válidas:", len(df))
print(df["action"].value_counts().to_string())
print("Fuentes distintas:", df["source_node"].nunique(), "| fuentes dentro del lote 1 (primeros 20 genes):",
      df["source_node"].isin(gq.obtener_genes("hypertrophy")[1][:20]).sum())
df.head(8)

Interacciones válidas: 37
action
stimulation    25
inhibition     12
Fuentes distintas: 19 | fuentes dentro del lote 1 (primeros 20 genes): 37


,source_node,target_node,action
0,ADRA1A,GNAQ,stimulation
1,ADRA1A,GNA11,stimulation
2,ADRA1A,PLCB1,stimulation
3,ADRA1A,PRKCA,stimulation
4,ADRA1B,GNAQ,stimulation
5,ADRA1B,GNA11,stimulation
6,ADRA1B,PLCB3,stimulation
7,ADRA1D,GNAQ,stimulation
